In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/telco-Customer-Churn.csv')

# Apply same fix as before
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)

print(f"Data loaded: {df.shape}")
print(df.head(2))

Data loaded: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   
1          No          No              No        One year               No   

      PaymentMethod MonthlyCharges  TotalCharges  Churn  
0  Electronic check          29.85         29.85     No  
1      Mailed check          56.95       1889.50     No  

[2 rows x 21 columns]


In [5]:
# Convert Yes/No to 1/0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Churn value counts:")
print(df['Churn'].value_counts())
print(f"\n0 = Not Churned, 1 = Churned")

Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64

0 = Not Churned, 1 = Churned


In [6]:
# How much has this customer paid per month on average?
# Low value = new customer, High value = loyal but expensive
df['avg_charge_per_tenure'] = df['TotalCharges'] / (df['tenure'] + 1)

print("Feature created: avg_charge_per_tenure")
print(df[['customerID', 'tenure', 'TotalCharges', 'avg_charge_per_tenure']].head(5))

Feature created: avg_charge_per_tenure
   customerID  tenure  TotalCharges  avg_charge_per_tenure
0  7590-VHVEG       1         29.85              14.925000
1  5575-GNVDE      34       1889.50              53.985714
2  3668-QPYBK       2        108.15              36.050000
3  7795-CFOCW      45       1840.75              40.016304
4  9237-HQITU       2        151.65              50.550000


In [7]:
# Month-to-month = highest risk, Two year = lowest risk
contract_risk = {
    'Month-to-month': 3,
    'One year': 2,
    'Two year': 1
}
df['contract_risk_score'] = df['Contract'].map(contract_risk)

print("Feature created: contract_risk_score")
print(df['contract_risk_score'].value_counts())

Feature created: contract_risk_score
contract_risk_score
3    3875
1    1695
2    1473
Name: count, dtype: int64


In [8]:
# Customers using more services are less likely to leave
service_cols = [
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

# Count how many services each customer uses
df['total_services'] = df[service_cols].apply(
    lambda row: sum(1 for val in row if val not in ['No', 'No internet service', 'No phone service']),
    axis=1
)

print("Feature created: total_services")
print(df['total_services'].value_counts().sort_index())

Feature created: total_services
total_services
1    1264
2     859
3     846
4     965
5     922
6     908
7     676
8     395
9     208
Name: count, dtype: int64


In [9]:
# Customers paying above average monthly charges
avg_charge = df['MonthlyCharges'].mean()
df['is_high_value'] = (df['MonthlyCharges'] > avg_charge).astype(int)

print(f"Average monthly charge: ${avg_charge:.2f}")
print(f"\nHigh value customers : {df['is_high_value'].sum()}")
print(f"Low value customers  : {(df['is_high_value']==0).sum()}")

Average monthly charge: $64.76

High value customers : 3923
Low value customers  : 3120


In [10]:
# Group customers into life stages
def tenure_group(tenure):
    if tenure <= 12:
        return 'New'          # 0-12 months
    elif tenure <= 24:
        return 'Developing'   # 1-2 years
    elif tenure <= 48:
        return 'Established'  # 2-4 years
    else:
        return 'Loyal'        # 4+ years

df['tenure_group'] = df['tenure'].apply(tenure_group)

print("Feature created: tenure_group")
print(df.groupby('tenure_group')['Churn'].mean().sort_values(ascending=False) * 100)
print("\nInsight: New customers churn the most — confirms our EDA finding")

Feature created: tenure_group
tenure_group
New            47.438243
Developing     28.710938
Established    20.388959
Loyal           9.513176
Name: Churn, dtype: float64

Insight: New customers churn the most — confirms our EDA finding


In [11]:
# Electronic check = highest churn risk from EDA
df['is_risky_payment'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

churn_rate = df.groupby('is_risky_payment')['Churn'].mean() * 100
print("Churn rate by payment risk:")
print(f"  Safe payment methods  : {churn_rate[0]:.1f}%")
print(f"  Electronic check      : {churn_rate[1]:.1f}%")

Churn rate by payment risk:
  Safe payment methods  : 17.1%
  Electronic check      : 45.3%


In [12]:
from sklearn.preprocessing import LabelEncoder

# Columns to encode
categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'tenure_group'
]

le = LabelEncoder()
df_encoded = df.copy()

for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

print("All categorical columns encoded ✓")
print(f"\nFinal dataset shape: {df_encoded.shape}")
print(f"New features added : avg_charge_per_tenure, contract_risk_score,")
print(f"                     total_services, is_high_value,")
print(f"                     tenure_group, is_risky_payment")

All categorical columns encoded ✓

Final dataset shape: (7043, 27)
New features added : avg_charge_per_tenure, contract_risk_score,
                     total_services, is_high_value,
                     tenure_group, is_risky_payment


In [13]:
# Drop customerID — not useful for modeling
df_encoded.drop('customerID', axis=1, inplace=True)

# Save processed data
df_encoded.to_csv('../data/processed/telecom_churn_processed.csv', index=False)

print("Processed dataset saved to data/processed/ ✓")
print(f"\nFinal shape  : {df_encoded.shape}")
print(f"Total features: {df_encoded.shape[1] - 1} features + 1 target")
print("\nReady for Modeling → Notebook 04")

Processed dataset saved to data/processed/ ✓

Final shape  : (7043, 26)
Total features: 25 features + 1 target

Ready for Modeling → Notebook 04
